We're building a custom dataset by combining ETF options chains from yfinance with earnings dates for each ETF's top holdings, scraped from stockanalysis.com. We verified scraping is allowed via their robots.txt. Our main question is whether implied volatility in ETF options responds to earnings events from the ETF's largest individual holdings

In [ ]:
import pandas as pd
import numpy as np
import requests
import re
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
import json

Use yfinance api and scrape stock analysis

In [ ]:
#Yfinance API
import yfinance as yf

#scrape data from stockanalysis.com
url = "https://stockanalysis.com/stocks/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', {'class': 'stock-table'})

Join yfinance options and history and stock analysis earnings for each stock in ETF

# Section 1 — Price History

In [1]:
# Imports

import pandas as pd
from data_collection import get_price_history

In [2]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]
PERIOD  = "6mo"
OUTPUT_CSV = "price_history.csv"

In [3]:
# Fetch price history for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching price history: {ticker} ...", end=" ")
    try:
        df = get_price_history(ticker, period=PERIOD)
        frames.append(df)
        print(f"✓  {len(df):,} rows")
    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

prices = pd.concat(frames, ignore_index=False)

Fetching price history: VOO ... 

INFO | get_price_history('VOO', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.


✓  125 rows
Fetching price history: QQQ ... 

INFO | get_price_history('QQQ', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.


✓  125 rows
Fetching price history: ARKQ ... 

INFO | get_price_history('ARKQ', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.


✓  125 rows
Fetching price history: BOTZ ... 

INFO | get_price_history('BOTZ', period='6mo'): 125 rows, 2025-10-01 to 2026-03-31.


✓  125 rows


In [4]:
# Clean and standardize

# Reset index so Date becomes a regular column
prices = prices.reset_index()

# Rename to match acceptance criteria exactly
prices = prices.rename(columns={"index": "Date", "ticker": "ticker"})

# Keep only required columns
KEEP_COLS = ["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
cols = [c for c in KEEP_COLS if c in prices.columns]
prices = prices[cols]

# Ensure correct dtypes
prices["Date"]   = pd.to_datetime(prices["Date"]).dt.tz_localize(None)
prices["Open"]   = pd.to_numeric(prices["Open"],   errors="coerce")
prices["High"]   = pd.to_numeric(prices["High"],   errors="coerce")
prices["Low"]    = pd.to_numeric(prices["Low"],    errors="coerce")
prices["Close"]  = pd.to_numeric(prices["Close"],  errors="coerce")
prices["Volume"] = pd.to_numeric(prices["Volume"], errors="coerce")

# Sort for readability
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)

prices.head(10)


Price,ticker,Date,Open,High,Low,Close,Volume
0,ARKQ,2025-10-01,110.815463,113.388857,110.715721,113.229263,262900
1,ARKQ,2025-10-02,115.174275,115.503432,114.176837,115.303940,297500
2,ARKQ,2025-10-03,116.361226,116.999586,114.605731,116.321327,445000
3,ARKQ,2025-10-06,120.051745,121.976802,119.782432,121.737419,704100
4,ARKQ,2025-10-07,121.996748,123.192682,119.263768,120.201363,708300
5,ARKQ,2025-10-08,120.271184,122.744828,119.847270,122.665039,349700
6,ARKQ,2025-10-09,122.505443,123.450016,120.131541,121.398285,304800
7,ARKQ,2025-10-10,121.498031,122.914400,115.483475,115.653038,626600
8,ARKQ,2025-10-13,119.124128,120.510567,117.967095,119.842285,360000
9,ARKQ,2025-10-14,117.588069,121.936903,115.433605,120.420799,319100


In [5]:
# Verify no major gaps in the time series

print("=== Gap Analysis ===\n")

for ticker in TICKERS:
    subset = prices[prices["ticker"] == ticker].copy()

    if subset.empty:
        print(f"{ticker}: No data.\n")
        continue

    # Expected trading days (roughly 5 per week, minus holidays)
    date_range = pd.bdate_range(subset["Date"].min(), subset["Date"].max())
    expected   = len(date_range)
    actual     = len(subset)
    missing    = expected - actual

    # Find specific gap dates
    actual_dates   = set(subset["Date"].dt.date)
    expected_dates = set(date_range.date)
    gap_dates      = sorted(expected_dates - actual_dates)

    print(f"{ticker}:")
    print(f"  Date range : {subset['Date'].min().date()} → {subset['Date'].max().date()}")
    print(f"  Expected   : {expected} trading days")
    print(f"  Actual     : {actual} rows")
    print(f"  Missing    : {missing} days", end="")

    if missing == 0:
        print(" ✓ No gaps")
    elif missing <= 5:
        print(f" — likely holidays: {gap_dates}")
    else:
        print(f" ⚠ Potential data issue — gap dates: {gap_dates[:10]}")
    print()

=== Gap Analysis ===

VOO:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — likely holidays: [datetime.date(2025, 11, 27), datetime.date(2025, 12, 25), datetime.date(2026, 1, 1), datetime.date(2026, 1, 19), datetime.date(2026, 2, 16)]

QQQ:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — likely holidays: [datetime.date(2025, 11, 27), datetime.date(2025, 12, 25), datetime.date(2026, 1, 1), datetime.date(2026, 1, 19), datetime.date(2026, 2, 16)]

ARKQ:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — likely holidays: [datetime.date(2025, 11, 27), datetime.date(2025, 12, 25), datetime.date(2026, 1, 1), datetime.date(2026, 1, 19), datetime.date(2026, 2, 16)]

BOTZ:
  Date range : 2025-10-01 → 2026-03-31
  Expected   : 130 trading days
  Actual     : 125 rows
  Missing    : 5 days — 

In [6]:
# Quick summary stats

summary = (
    prices.groupby("ticker")
    .agg(
        rows        = ("Close", "count"),
        start_date  = ("Date",  "min"),
        end_date    = ("Date",  "max"),
        avg_close   = ("Close", "mean"),
        min_close   = ("Close", "min"),
        max_close   = ("Close", "max"),
        avg_volume  = ("Volume","mean"),
    )
    .round(2)
)
print(summary)


        rows start_date   end_date  avg_close  min_close  max_close  \
ticker                                                                
ARKQ     125 2025-10-01 2026-03-31     118.94     101.00     134.05   
BOTZ     125 2025-10-01 2026-03-31      36.52      31.99      39.66   
QQQ      125 2025-10-01 2026-03-31     609.55     558.28     634.15   
VOO      125 2025-10-01 2026-03-31     620.67     580.93     637.69   

         avg_volume  
ticker               
ARKQ      291031.46  
BOTZ      878899.14  
QQQ     61094980.79  
VOO      9861425.14  


/var/folders/v1/_0s9cgvs7f50tdjbmzh49yn00000gn/T/ipykernel_31397/1166243420.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


In [7]:
# Save to CSV

prices.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved CSV → {OUTPUT_CSV}  ({prices.shape[0]:,} rows, {prices.shape[1]} columns)")



Saved CSV → price_history.csv  (500 rows, 7 columns)


In [8]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV, parse_dates=["Date"])
assert reloaded.shape == prices.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows, {reloaded.shape[1]} columns.")
reloaded.head()

Reload check passed — 500 rows, 7 columns.


,ticker,Date,Open,High,Low,Close,Volume
0,ARKQ,2025-10-01,110.815463,113.388857,110.715721,113.229263,262900
1,ARKQ,2025-10-02,115.174275,115.503432,114.176837,115.303940,297500
2,ARKQ,2025-10-03,116.361226,116.999586,114.605731,116.321327,445000
3,ARKQ,2025-10-06,120.051745,121.976802,119.782432,121.737419,704100
4,ARKQ,2025-10-07,121.996748,123.192682,119.263768,120.201363,708300


# Section 2 — Options Chain

In [ ]:
# Imports 

import pandas as pd
from data_collection import get_options_chain

In [ ]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]

KEEP_COLS = [
    "ticker",
    "expiration",
    "contractType",   # renamed from option_type for clarity
    "strike",
    "bid",
    "ask",
    "volume",
    "openInterest",
    "impliedVolatility",
]

OUTPUT_CSV     = "options_chain.csv"
OUTPUT_PARQUET = "options_chain.parquet"

In [ ]:
# Fetch options chain for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching options chain: {ticker} ...", end=" ")
    try:
        df = get_options_chain(ticker)

        # Rename option_type -> contractType to match acceptance criteria
        df = df.rename(columns={"option_type": "contractType"})

        # Keep only the required columns (drop any that don't exist gracefully)
        cols = [c for c in KEEP_COLS if c in df.columns]
        df = df[cols]

        frames.append(df)
        print(f"✓  {len(df):,} rows")

    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

options = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows collected: {len(options):,}")

In [ ]:
# Light cleaning

# Ensure correct dtypes
options["expiration"]       = pd.to_datetime(options["expiration"])
options["strike"]           = pd.to_numeric(options["strike"],           errors="coerce")
options["bid"]              = pd.to_numeric(options["bid"],              errors="coerce")
options["ask"]              = pd.to_numeric(options["ask"],              errors="coerce")
options["volume"]           = pd.to_numeric(options["volume"],           errors="coerce")
options["openInterest"]     = pd.to_numeric(options["openInterest"],     errors="coerce")
options["impliedVolatility"]= pd.to_numeric(options["impliedVolatility"],errors="coerce")

# Drop rows where strike or IV is missing — not useful for analysis
before = len(options)
options = options.dropna(subset=["strike", "impliedVolatility"])
dropped = before - len(options)
if dropped:
    print(f"Dropped {dropped:,} rows with missing strike or impliedVolatility.")

# Sort for readability
options = options.sort_values(
    ["ticker", "expiration", "contractType", "strike"]
).reset_index(drop=True)

options.head(10)


In [ ]:
# Quick summary

summary = (
    options.groupby(["ticker", "contractType"])
    .agg(
        expirations   = ("expiration", "nunique"),
        contracts     = ("strike",     "count"),
        avg_iv        = ("impliedVolatility", "mean"),
        total_oi      = ("openInterest", "sum"),
    )
    .round(4)
)
print(summary)

In [ ]:
# Save to disk

options.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV     → {OUTPUT_CSV}  ({options.shape[0]:,} rows)")

# Section 3 — Spot Price Join

# Section 4 — Holdings

# Section 5 — Earnings